# **DATA COLECTION**

## **1. Import Libraries**

In [1]:
import os
import sys
import json
import time
import subprocess
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 120)

## **2. Identify the data source**

### Nguồn 1: Tiki API
- Endpoint danh sách: https://tiki.vn/api/v2/products
- Endpoint chi tiết: https://tiki.vn/api/v2/products/{product_id}
- Cơ chế hiện có trong script: thu thập ID theo category + keyword, sau đó gọi API chi tiết, lưu checkpoint để resume

### Nguồn 2: eBay Browse API
- OAuth token endpoint: https://api.ebay.com/identity/v1/oauth2/token
- Browse search endpoint: https://api.ebay.com/buy/browse/v1/item_summary/search
- Cơ chế hiện có trong script: crawl theo nhiều query, ghép kết quả và loại trùng theo item_id

### Lưu ý vận hành
- Tôn trọng rate limit, thêm delay giữa các request
- eBay yêu cầu APP_ID/CERT_ID hợp lệ
- Tiki crawler có checkpoint giúp chạy nhiều lần để đạt target lớn

In [2]:
# Standard project paths
PROJECT_ROOT = Path("..").resolve()
CRAWLER_DIR = PROJECT_ROOT / "src" / "crawlers"
DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
TIKI_RUN_DIR = DATA_RAW_DIR  # Canonical working directory for Tiki outputs/checkpoint

TIKI_SCRIPT = CRAWLER_DIR / "tiki_crawler.py"
EBAY_SCRIPT = CRAWLER_DIR / "ebay_crawler.py"

# Crawl run configuration
CRAWL_CONFIG = {
    "run_tiki": True,          # Set to True to run the Tiki crawler
    "run_ebay": False,          # Set to True to run the eBay crawler
    "tiki_target": 10000,       # Target number of rows for Tiki
    "tiki_max_pages": 120,      # Maximum pages per source (category/keyword)
    "run_timeout": 60 * 60 * 6, # Timeout for each crawl process (seconds)
    "tiki_reset_checkpoint": False  # True => remove checkpoint/backup/output before crawling
}

# eBay credentials: prefer environment variables to avoid hard-coding
EBAY_APP_ID = os.getenv("EBAY_APP_ID")
EBAY_CERT_ID = os.getenv("EBAY_CERT_ID")

print("Project root:", PROJECT_ROOT)
print("Crawler directory:", CRAWLER_DIR)
print("Raw data directory:", DATA_RAW_DIR)
print("Tiki run directory:", TIKI_RUN_DIR)
print("Tiki script exists:", TIKI_SCRIPT.exists())
print("eBay script exists:", EBAY_SCRIPT.exists())

Project root: E:\ĐHK23\NAM4\TQH\DV_Lab1
Crawler directory: E:\ĐHK23\NAM4\TQH\DV_Lab1\src\crawlers
Raw data directory: E:\ĐHK23\NAM4\TQH\DV_Lab1\data\raw
Tiki run directory: E:\ĐHK23\NAM4\TQH\DV_Lab1\data\raw
Tiki script exists: True
eBay script exists: True


## **3. Design data structures according to each eBay/Tiki source**

In [3]:
# Source-specific schemas (derived from the two existing crawlers)
tiki_columns = [
    "id", "name", "price", "original_price", "discount_rate", "rating_average",
    "review_count", "quantity_sold", "brand", "category", "url_key"
]

ebay_columns = [
    "query", "item_id", "legacy_item_id", "item_web_url", "item_href",
    "title", "subtitle", "price", "currency", "shipping_cost", "shipping_currency",
    "has_free_shipping", "total_cost", "condition", "condition_id", "buying_options",
    "top_rated_buying_experience", "priority_listing", "adult_only",
    "item_creation_date", "item_end_date", "seller_username", "seller_feedback_score",
    "seller_feedback_percent", "category_id", "category_path", "leaf_category_ids",
    "item_location_country", "item_location_postal", "image_url", "thumbnail_url"
]

script_info = {
    "tiki_crawler": {
        "entry_file": str(TIKI_SCRIPT),
        "list_endpoint": "https://tiki.vn/api/v2/products",
        "detail_endpoint": "https://tiki.vn/api/v2/products/{product_id}",
        "checkpoint_file": "tiki_checkpoint.json",
        "backup_file": "backup.csv",
        "default_output": "products.csv",
        "arguments": ["--target", "--max-pages"],
    },
    "ebay_crawler": {
        "entry_file": str(EBAY_SCRIPT),
        "oauth_endpoint": "https://api.ebay.com/identity/v1/oauth2/token",
        "search_endpoint": "https://api.ebay.com/buy/browse/v1/item_summary/search",
        "default_output": "ebay_products.csv",
        "credentials_required": ["APP_ID", "CERT_ID"],
        "collection_strategy": "multi-query + deduplicate by item_id",
    },
}

print("Tiki schema (", len(tiki_columns), "columns):")
print(tiki_columns)
print("\neBay schema (", len(ebay_columns), "columns):")
print(ebay_columns)
print("\nDetailed information by crawler script:")
print(json.dumps(script_info, indent=2, ensure_ascii=False))

Tiki schema ( 11 columns):
['id', 'name', 'price', 'original_price', 'discount_rate', 'rating_average', 'review_count', 'quantity_sold', 'brand', 'category', 'url_key']

eBay schema ( 31 columns):
['query', 'item_id', 'legacy_item_id', 'item_web_url', 'item_href', 'title', 'subtitle', 'price', 'currency', 'shipping_cost', 'shipping_currency', 'has_free_shipping', 'total_cost', 'condition', 'condition_id', 'buying_options', 'top_rated_buying_experience', 'priority_listing', 'adult_only', 'item_creation_date', 'item_end_date', 'seller_username', 'seller_feedback_score', 'seller_feedback_percent', 'category_id', 'category_path', 'leaf_category_ids', 'item_location_country', 'item_location_postal', 'image_url', 'thumbnail_url']

Detailed information by crawler script:
{
  "tiki_crawler": {
    "entry_file": "E:\\ĐHK23\\NAM4\\TQH\\DV_Lab1\\src\\crawlers\\tiki_crawler.py",
    "list_endpoint": "https://tiki.vn/api/v2/products",
    "detail_endpoint": "https://tiki.vn/api/v2/products/{product

## **4. Data collection function**

**Lưu ý:**
- Cập nhật API của eBay APP_ID/CERT_ID trước khi chạy.
- Tiki script hỗ trợ resume bằng checkpoint (tiki_checkpoint.json).
- Nên bật từng nguồn một để dễ theo dõi log.

In [4]:
def run_command(command, cwd, timeout_seconds):
    """Run a shell command and stream the captured output."""
    print("Command:", " ".join(command))
    print("Working directory:", cwd)

    start = time.time()
    completed = subprocess.run(
        command,
        cwd=str(cwd),
        text=True,
        capture_output=True,
        timeout=timeout_seconds,
        check=False,
    )
    elapsed = time.time() - start

    print("=" * 80)
    print("STDOUT:")
    print(completed.stdout if completed.stdout else "<empty>")
    print("=" * 80)
    print("STDERR:")
    print(completed.stderr if completed.stderr else "<empty>")
    print("=" * 80)
    print(f"Exit code: {completed.returncode} | Elapsed: {elapsed:.2f}s")

    return completed.returncode


def run_tiki_crawler(target=10000, max_pages=120, timeout_seconds=7200):
    if not TIKI_SCRIPT.exists():
        raise FileNotFoundError(f"File not found: {TIKI_SCRIPT}")

    TIKI_RUN_DIR.mkdir(parents=True, exist_ok=True)

    if CRAWL_CONFIG.get("tiki_reset_checkpoint", False):
        for stale_file in ["tiki_checkpoint.json", "backup.csv", "products.csv"]:
            stale_path = TIKI_RUN_DIR / stale_file
            if stale_path.exists():
                stale_path.unlink()
                print(f"[INFO] Removed stale file: {stale_path}")

    command = [
        sys.executable,
        str(TIKI_SCRIPT),
        "--target", str(target),
        "--max-pages", str(max_pages),
    ]
    return run_command(command, cwd=TIKI_RUN_DIR, timeout_seconds=timeout_seconds)


def run_ebay_crawler(timeout_seconds=7200):
    if not EBAY_SCRIPT.exists():
        raise FileNotFoundError(f"File not found: {EBAY_SCRIPT}")

    # Warn when credentials are not configured yet
    if not EBAY_APP_ID or not EBAY_CERT_ID:
        print("[WARN] EBAY_APP_ID / EBAY_CERT_ID were not found in environment variables.")
        print("[WARN] Update APP_ID/CERT_ID in ebay_crawler.py or export env vars before running.")

    command = [sys.executable, str(EBAY_SCRIPT)]
    return run_command(command, cwd=PROJECT_ROOT, timeout_seconds=timeout_seconds)

## **5. Implement data collection**

Bật/tắt từng nguồn bằng CRAWL_CONFIG ở cell cấu hình. Khi chạy thật, nên chạy theo thứ tự Tiki -> eBay để dễ kiểm tra output.

In [ ]:
run_log = {
    "started_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "tiki_exit_code": None,
    "ebay_exit_code": None,
}

if CRAWL_CONFIG["run_tiki"]:
    print("\n" + "=" * 100)
    print("STARTING TIKI CRAWL")
    print("=" * 100)
    run_log["tiki_exit_code"] = run_tiki_crawler(
        target=CRAWL_CONFIG["tiki_target"],
        max_pages=CRAWL_CONFIG["tiki_max_pages"],
        timeout_seconds=CRAWL_CONFIG["run_timeout"],
    )
else:
    print("[INFO] Skipping Tiki crawl (run_tiki=False)")

if CRAWL_CONFIG["run_ebay"]:
    print("\n" + "=" * 100)
    print("STARTING EBAY CRAWL")
    print("=" * 100)
    run_log["ebay_exit_code"] = run_ebay_crawler(
        timeout_seconds=CRAWL_CONFIG["run_timeout"],
    )
else:
    print("[INFO] Skipping eBay crawl (run_ebay=False)")

run_log["finished_at"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print("\nRun log:")
print(json.dumps(run_log, indent=2, ensure_ascii=False))


STARTING TIKI CRAWL
Command: c:\Users\hp\AppData\Local\Programs\Python\Python313\python.exe E:\ĐHK23\NAM4\TQH\DV_Lab1\src\crawlers\tiki_crawler.py --target 10000 --max-pages 120
Working directory: E:\ĐHK23\NAM4\TQH\DV_Lab1\data\raw


## **6. Convert to DataFrame**

In [6]:
def read_csv_with_fallback(paths, source_name):
    for path in paths:
        if path.exists():
            df = pd.read_csv(path)
            print(f"[{source_name}] Loaded {len(df):,} rows from: {path}")
            return df, path
    print(f"[{source_name}] Data file not found.")
    return pd.DataFrame(), None

expected_tiki_output = TIKI_RUN_DIR / "products.csv"

tiki_candidates = [
    expected_tiki_output,
    DATA_RAW_DIR / "tiki_products.csv",
    PROJECT_ROOT / "products.csv",
    PROJECT_ROOT / "backup.csv",
]

ebay_candidates = [
    DATA_RAW_DIR / "ebay_products.csv",
    PROJECT_ROOT / "ebay_products.csv",
]

df_tiki, tiki_source_path = read_csv_with_fallback(tiki_candidates, "Tiki")
df_ebay, ebay_source_path = read_csv_with_fallback(ebay_candidates, "eBay")

if tiki_source_path is not None and tiki_source_path.resolve() != expected_tiki_output.resolve():
    print("[WARN] Tiki data was loaded from a non-canonical file.")
    print(f"[WARN] Expected: {expected_tiki_output}")
    print(f"[WARN] Actual:   {tiki_source_path}")
    print("[WARN] This usually means stale outputs/checkpoints from another working directory.")

print("\nDataset shapes:")
print("- Tiki:", df_tiki.shape)
print("- eBay:", df_ebay.shape)

[Tiki] Loaded 10,000 rows from: C:\Users\admin\OneDrive\Desktop\visualize\DV_Lab1\data\raw\products.csv
[eBay] Loaded 15,744 rows from: C:\Users\admin\OneDrive\Desktop\visualize\DV_Lab1\data\raw\ebay_products.csv

Dataset shapes:
- Tiki: (10000, 11)
- eBay: (15744, 31)


## **7. Initial data quality check**

Kiểm tra gồm: cấu trúc cột, thống kê số học, missing values, và trùng lặp theo khóa chính của từng nguồn.

In [7]:
def profile_dataframe(df, name, max_numeric_cols=12):
    print("\n" + "=" * 100)
    print(f"[{name}] OVERVIEW")
    print("=" * 100)
    print("Shape:", df.shape)

    if df.empty:
        print("DataFrame is empty, skipping profiling.")
        return

    print("\nColumns:")
    print(df.columns.tolist())

    print("\nInfo:")
    df.info()

    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_cols:
        use_cols = numeric_cols[:max_numeric_cols]
        print(f"\nNumeric describe ({len(use_cols)}/{len(numeric_cols)} columns):")
        display(df[use_cols].describe().T)
    else:
        print("\nNo numeric columns available for describe().")

profile_dataframe(df_tiki, "Tiki")
profile_dataframe(df_ebay, "eBay")


[Tiki] OVERVIEW
Shape: (10000, 11)

Columns:
['id', 'name', 'price', 'original_price', 'discount_rate', 'rating_average', 'review_count', 'quantity_sold', 'brand', 'category', 'url_key']

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id              10000 non-null  int64  
 1   name            10000 non-null  object 
 2   price           10000 non-null  int64  
 3   original_price  10000 non-null  int64  
 4   discount_rate   10000 non-null  int64  
 5   rating_average  10000 non-null  float64
 6   review_count    10000 non-null  int64  
 7   quantity_sold   10000 non-null  int64  
 8   brand           10000 non-null  object 
 9   category        10000 non-null  object 
 10  url_key         10000 non-null  object 
dtypes: float64(1), int64(6), object(4)
memory usage: 859.5+ KB

Numeric describe (7/7 columns):


,count,mean,std,min,25%,50%,75%,max
id,10000.0,1.979565e+08,9.692036e+07,126003.0,114809885.0,263309157.5,2.767340e+08,279280455.0
price,10000.0,5.039067e+05,1.337487e+06,1000.0,89600.0,173425.0,3.765642e+05,42250000.0
original_price,10000.0,5.832217e+05,1.508766e+06,1000.0,99000.0,193100.0,4.241500e+05,42250000.0
discount_rate,10000.0,9.746300e+00,1.507456e+01,0.0,0.0,0.0,1.900000e+01,77.0
rating_average,10000.0,2.856690e+00,2.331482e+00,0.0,0.0,4.5,5.000000e+00,5.0
review_count,10000.0,4.252330e+01,2.353843e+02,0.0,0.0,1.0,1.000000e+01,7116.0
quantity_sold,10000.0,6.515299e+02,5.580116e+03,0.0,2.0,13.0,9.200000e+01,284260.0



[eBay] OVERVIEW
Shape: (15744, 31)

Columns:
['query', 'item_id', 'legacy_item_id', 'item_web_url', 'item_href', 'title', 'subtitle', 'price', 'currency', 'shipping_cost', 'shipping_currency', 'has_free_shipping', 'total_cost', 'condition', 'condition_id', 'buying_options', 'top_rated_buying_experience', 'priority_listing', 'adult_only', 'item_creation_date', 'item_end_date', 'seller_username', 'seller_feedback_score', 'seller_feedback_percent', 'category_id', 'category_path', 'leaf_category_ids', 'item_location_country', 'item_location_postal', 'image_url', 'thumbnail_url']

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15744 entries, 0 to 15743
Data columns (total 31 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   query                        15744 non-null  object 
 1   item_id                      15744 non-null  object 
 2   legacy_item_id               15744 non-null  int64  
 3   item_w

,count,mean,std,min,25%,50%,75%,max
legacy_item_id,15744.0,2.649642e+11,9.113163e+10,1.112208e+11,1.780135e+11,2.674966e+11,3.556178e+11,4.068256e+11
subtitle,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
price,15744.0,1.453998e+02,3.391169e+02,9.900000e-01,3.533000e+01,7.999000e+01,1.750000e+02,3.417302e+04
shipping_cost,15744.0,1.472739e+00,6.397976e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.699000e+02
total_cost,15744.0,1.468725e+02,3.394541e+02,9.900000e-01,3.648000e+01,7.999000e+01,1.798500e+02,3.417302e+04
condition_id,15743.0,1.899348e+03,1.042950e+03,1.000000e+03,1.000000e+03,1.500000e+03,3.000000e+03,7.000000e+03
seller_feedback_score,15744.0,5.485893e+04,1.634012e+05,-1.000000e+00,6.010000e+02,4.757000e+03,3.173600e+04,3.329896e+06
seller_feedback_percent,15744.0,9.685243e+01,1.502385e+01,0.000000e+00,9.900000e+01,9.960000e+01,9.990000e+01,1.000000e+02
category_id,15744.0,9.541612e+04,6.692916e+04,1.640000e+02,9.355000e+03,1.125290e+05,1.714850e+05,2.631250e+05


In [8]:
def missing_report(df, name):
    print("\n" + "-" * 100)
    print(f"[{name}] MISSING VALUES")
    print("-" * 100)

    if df.empty:
        print("DataFrame is empty.")
        return pd.DataFrame(columns=["missing_count", "missing_pct"])

    missing_count = df.isna().sum()
    missing_pct = (missing_count / len(df) * 100).round(2)
    report = pd.DataFrame({
        "missing_count": missing_count,
        "missing_pct": missing_pct
    }).sort_values("missing_count", ascending=False)

    report = report[report["missing_count"] > 0]
    if report.empty:
        print("No missing values found.")
    else:
        display(report)
    return report

missing_tiki = missing_report(df_tiki, "Tiki")
missing_ebay = missing_report(df_ebay, "eBay")


----------------------------------------------------------------------------------------------------
[Tiki] MISSING VALUES
----------------------------------------------------------------------------------------------------
No missing values found.

----------------------------------------------------------------------------------------------------
[eBay] MISSING VALUES
----------------------------------------------------------------------------------------------------


,missing_count,missing_pct
subtitle,15744,100.00
item_end_date,15718,99.83
shipping_currency,2130,13.53
item_location_postal,1523,9.67
thumbnail_url,6,0.04
image_url,6,0.04
condition_id,1,0.01


In [9]:
def duplicate_report(df, name, key_col):
    print("\n" + "-" * 100)
    print(f"[{name}] DUPLICATE CHECK by {key_col}")
    print("-" * 100)

    if df.empty:
        print("DataFrame is empty.")
        return df

    if key_col not in df.columns:
        print(f"Key column not found: {key_col}")
        return df

    dup_count = df.duplicated(subset=[key_col]).sum()
    print(f"Number of duplicate key rows: {dup_count}")

    if dup_count > 0:
        print("Removing duplicates and keeping the first occurrence.")
        dedup_df = df.drop_duplicates(subset=[key_col], keep="first")
        print("Shape before/after:", df.shape, "->", dedup_df.shape)
        return dedup_df

    return df

df_tiki = duplicate_report(df_tiki, "Tiki", "id")
df_ebay = duplicate_report(df_ebay, "eBay", "item_id")


----------------------------------------------------------------------------------------------------
[Tiki] DUPLICATE CHECK by id
----------------------------------------------------------------------------------------------------
Number of duplicate key rows: 0

----------------------------------------------------------------------------------------------------
[eBay] DUPLICATE CHECK by item_id
----------------------------------------------------------------------------------------------------
Number of duplicate key rows: 0


## **8. Saved Raw Data**

Lưu dữ liệu theo từng nguồn:
- data/raw/tiki_products.csv
- data/raw/ebay_products.csv

In [10]:
DATA_RAW_DIR.mkdir(parents=True, exist_ok=True)

tiki_output = DATA_RAW_DIR / "tiki_products.csv"
ebay_output = DATA_RAW_DIR / "ebay_products.csv"

save_report = {
    "saved_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "tiki": {
        "source_file": str(tiki_source_path) if tiki_source_path else None,
        "output_file": str(tiki_output),
        "rows": 0,
        "columns": 0,
    },
    "ebay": {
        "source_file": str(ebay_source_path) if ebay_source_path else None,
        "output_file": str(ebay_output),
        "rows": 0,
        "columns": 0,
    },
}

if not df_tiki.empty:
    df_tiki.to_csv(tiki_output, index=False, encoding="utf-8-sig")
    save_report["tiki"]["rows"] = int(df_tiki.shape[0])
    save_report["tiki"]["columns"] = int(df_tiki.shape[1])
    print(f"Saved Tiki raw data: {tiki_output} ({len(df_tiki):,} rows)")
else:
    print("[INFO] No Tiki data available to save.")

if not df_ebay.empty:
    df_ebay.to_csv(ebay_output, index=False, encoding="utf-8-sig")
    save_report["ebay"]["rows"] = int(df_ebay.shape[0])
    save_report["ebay"]["columns"] = int(df_ebay.shape[1])
    print(f"Saved eBay raw data: {ebay_output} ({len(df_ebay):,} rows)")
else:
    print("[INFO] No eBay data available to save.")

print("\nData saving report:")
print(json.dumps(save_report, indent=2, ensure_ascii=False))

if not df_tiki.empty:
    print("\n[Tiki] Preview:")
    display(df_tiki.head())

if not df_ebay.empty:
    print("\n[eBay] Preview:")
    display(df_ebay.head())

Saved Tiki raw data: C:\Users\admin\OneDrive\Desktop\visualize\DV_Lab1\data\raw\tiki_products.csv (10,000 rows)
Saved eBay raw data: C:\Users\admin\OneDrive\Desktop\visualize\DV_Lab1\data\raw\ebay_products.csv (15,744 rows)

Data saving report:
{
  "saved_at": "2026-04-14 21:34:03",
  "tiki": {
    "source_file": "C:\\Users\\admin\\OneDrive\\Desktop\\visualize\\DV_Lab1\\data\\raw\\products.csv",
    "output_file": "C:\\Users\\admin\\OneDrive\\Desktop\\visualize\\DV_Lab1\\data\\raw\\tiki_products.csv",
    "rows": 10000,
    "columns": 11
  },
  "ebay": {
    "source_file": "C:\\Users\\admin\\OneDrive\\Desktop\\visualize\\DV_Lab1\\data\\raw\\ebay_products.csv",
    "output_file": "C:\\Users\\admin\\OneDrive\\Desktop\\visualize\\DV_Lab1\\data\\raw\\ebay_products.csv",
    "rows": 15744,
    "columns": 31
  }
}

[Tiki] Preview:


,id,name,price,original_price,discount_rate,rating_average,review_count,quantity_sold,brand,category,url_key
0,146417196,1 hộp(500g) Ngũ Cốc Min Min Lợi Sữa 38 Loại Hạt Chuyên Lợi Sữa và Phục Hồi Sức Khỏe Sau Sinh,270000,270000,0,4.5,2,5,Min Min,Bách Hóa Online,1-hop-500g-ngu-coc-min-min-loi-sua-30hat-chuyen-loi-sua-va-phuc-hoi-suc-khoe-sau-sinh-p146417196
1,165025828,(Nghe To) Loa Cộng Hưởng Khuếch Đại Âm Thanh KhoNCC Hàng Chính Hãng - Vừa Giá Đỡ Điện Thoại Chắc Chắn - KPD-1087-LoaCH,188000,188000,0,1.0,1,12,KHONCC,Loa Nghe Nhạc,nghe-to-loa-cong-huong-khuech-dai-am-thanh-khoncc-hang-chinh-hang-vua-gia-do-dien-thoai-chac-chan-kpd-1087-loach-p16...
2,178058698,Áo Len Nữ Cardigan Kiểu Dài Thời Trang Xinh ALD01 MayHomes,84000,99000,15,4.5,2,24,MAYHOMES,Thời trang nữ,ao-len-nu-cardigan-kieu-dai-thoi-trang-xinh-ald01-mayhomes-p178058698
3,276173692,"Ốp Lưng Siêu Mỏng Nhám cho iPhone 16 Pro/ 16 Pro Max/ 16/ 16 Plus MIPOW PREMIUM SLIM CASE, chống vân tay_ Hàng chính...",280000,280000,0,5.0,2,12,MIPOW,Thiết Bị Số - Phụ Kiện Số,op-lung-sieu-mong-nham-cho-iphone-16-pro-16-pro-max-16-16-plus-mipow-premium-slim-case-chong-van-tay_-hang-chinh-han...
4,276289489,Bộ sách Kỳ thi năng lực Nhật ngữ - Point & Practice N4,53000,75000,29,5.0,2,80,No Brand,Root,bo-sach-ky-thi-nang-luc-nhat-ngu-point-practice-n4-p276289489



[eBay] Preview:


,query,item_id,legacy_item_id,item_web_url,item_href,title,subtitle,price,currency,shipping_cost,shipping_currency,has_free_shipping,total_cost,condition,condition_id,buying_options,top_rated_buying_experience,priority_listing,adult_only,item_creation_date,item_end_date,seller_username,seller_feedback_score,seller_feedback_percent,category_id,category_path,leaf_category_ids,item_location_country,item_location_postal,image_url,thumbnail_url
0,laptop,v1|276709583737|0,276709583737,https://www.ebay.com/itm/276709583737?_skw=laptop&hash=item406d2d2b79:g:m3YAAeSwJJZpOyMs,https://api.ebay.com/buy/browse/v1/item/v1%7C276709583737%7C0,"Acer Swift Go 14"" Laptop Intel Core Ultra 7 155H 16GB RAM 1TB SSD Refurbished",NaN,371.99,USD,0.0,USD,True,371.99,Certified - Refurbished,2000.0,FIXED_PRICE,False,False,False,2024-10-30T23:21:59.000Z,NaN,acer,117605,99.0,177,PC Laptops & Netbooks,177,US,785**,https://i.ebayimg.com/images/g/m3YAAeSwJJZpOyMs/s-l225.jpg,https://i.ebayimg.com/images/g/m3YAAeSwJJZpOyMs/s-l1600.jpg
1,laptop,v1|256696935114|0,256696935114,https://www.ebay.com/itm/256696935114?_skw=laptop&hash=item3bc45462ca:g:CuUAAeSwxGxpw-Qi&amdata=enc%3AAQALAAAA4ACCtX...,https://api.ebay.com/buy/browse/v1/item/v1%7C256696935114%7C0,"HP ProBook Touchscreen Laptop 11.6"" Core i3 8GB RAM 128GB SSD Win 11 Pro HDMI",NaN,146.11,USD,0.0,USD,True,146.11,Good - Refurbished,2030.0,FIXED_PRICE,False,True,False,2024-10-31T15:10:17.000Z,NaN,discountcomputerdepot,156372,99.3,177,PC Laptops & Netbooks,177,US,757**,https://i.ebayimg.com/images/g/CuUAAeSwxGxpw-Qi/s-l225.jpg,https://i.ebayimg.com/images/g/CuUAAeSwxGxpw-Qi/s-l1600.jpg
2,laptop,v1|286288847506|0,286288847506,https://www.ebay.com/itm/286288847506?_skw=laptop&hash=item42a8252292:g:Mq8AAeSwIQ9puPl-&amdata=enc%3AAQALAAAA4ACCtX...,https://api.ebay.com/buy/browse/v1/item/v1%7C286288847506%7C0,Lenovo ThinkPad E15 15.6” HD Laptop AMD Ryzen 5 16GB RAM 256GB SSD Windows 11,NaN,254.75,USD,0.0,USD,True,254.75,Good - Refurbished,2030.0,FIXED_PRICE,False,True,False,2025-01-28T14:21:11.000Z,NaN,discountcomputerdepot,156372,99.3,177,PC Laptops & Netbooks,177,US,757**,https://i.ebayimg.com/images/g/Mq8AAeSwIQ9puPl-/s-l225.jpg,https://i.ebayimg.com/images/g/Mq8AAeSwIQ9puPl-/s-l1200.jpg
3,laptop,v1|277352801036|0,277352801036,https://www.ebay.com/itm/277352801036?_skw=laptop&hash=item409383e30c:g:vHUAAeSwg0ZphMv7,https://api.ebay.com/buy/browse/v1/item/v1%7C277352801036%7C0,Acer Predator Helios Neo AI 16S Gaming Laptop RTX 5070 Ti Certified Refurbished,NaN,1275.95,USD,0.0,USD,True,1275.95,Certified - Refurbished,2000.0,FIXED_PRICE,False,False,False,2025-08-29T21:15:09.000Z,NaN,acer,117605,99.0,177,PC Laptops & Netbooks,177,US,785**,https://i.ebayimg.com/images/g/vHUAAeSwg0ZphMv7/s-l225.jpg,https://i.ebayimg.com/images/g/vHUAAeSwg0ZphMv7/s-l960.jpg
4,laptop,v1|205814283029|0,205814283029,https://www.ebay.com/itm/205814283029?_skw=laptop&hash=item2feb7cbb15:g:THgAAeSwcVRpzCOo,https://api.ebay.com/buy/browse/v1/item/v1%7C205814283029%7C0,"ASUS - Vivobook 14 14"" FHD Laptop - Intel Core 5 120U with 8GB Memory - 256GB...",NaN,329.99,USD,0.0,USD,True,329.99,New,1000.0,FIXED_PRICE,False,False,False,2025-10-29T13:25:40.000Z,NaN,best_buy,952756,98.6,177,PC Laptops & Netbooks,177,US,NaN,https://i.ebayimg.com/images/g/THgAAeSwcVRpzCOo/s-l225.jpg,https://i.ebayimg.com/images/g/THgAAeSwcVRpzCOo/s-l1600.jpg


## **9. Summary**

Notebook 01 đã hoàn thiện theo luồng làm việc:
1. Cấu hình nguồn và tham số thu thập
2. Gọi script crawl tương ứng cho từng nguồn
3. Đọc dữ liệu thô riêng theo nguồn
4. Kiểm tra chất lượng ban đầu (missing, duplicate, profile) riêng theo nguồn
5. Lưu output riêng vào data/raw

**Output chính:**
- data/raw/tiki_products.csv
- data/raw/ebay_products.csv

**Ghi chú theo từng script:**
- Tiki: có checkpoint + backup, phù hợp chạy nhiều lần để đạt target lớn.
- eBay: cần credential hợp lệ, thu thập theo nhiều query và loại trùng theo item_id.

**Bước tiếp theo:**
- Notebook 02: tiền xử lý riêng cho từng nguồn.